# Train rigged-royale-matchup-ml on Kaggle GPU

**This version adds four generalization features** for synergies / archetypes / counters:
- **deck self-attention + archetype `[CLS]` token** — deck style (beatdown/cycle/bait/siege) and 3+ card combos, beyond pairwise synergy;
- **multi-head bilinear counters** (`cross_heads`) — several oriented "A counters B" modes (anti-air / anti-swarm / anti-tank) in parallel;
- **card2vec warm-start** — card embeddings initialised from self-supervised deck co-occurrence (PPMI+SVD), so rare cards start near functional neighbours;
- **inverse-frequency loss weighting** — rare cards/matchups aren't drowned out by the popular meta.

Still present: per-card elixir, evolved/hero identity, champion/hero roles.

⚠️ **RE-PREPARE REQUIRED — retrain from scratch.** The trophy buckets changed (seasonal road split at 14000) so the segment vocabulary differs, and old checkpoints' embeddings won't align. Re-run `prepare --overwrite` locally on freshly collected shards, then upload that `prepared` dataset. Old prepared datasets are **not** compatible.

This notebook auto-generates `card2vec.npy` + `card_frequencies.json` into `/kaggle/working` (the input dataset is read-only), so a freshly-prepared upload works even if those two files aren't in it.

**Before running:**
1. Right panel → **Settings** → **Accelerator** → **GPU T4 x2** (or P100).
2. Right panel → **Settings** → **Internet** → **On** (needed for `git clone`; account must be phone-verified).
3. Right panel → **Input** → **Add Input** → attach your **re-prepared** `prepared` dataset.

Then `Run All`.

In [ ]:
# 1. Clone code + verify GPU
!git clone -q https://github.com/mael-guimoyas/rigged-royale-matchup-ml.git
import torch
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU OFF — Settings → Accelerator → GPU T4"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Auto-locate the prepared data (finds manifest.json under /kaggle/input)
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "prepared data not found — attach your dataset via Input → Add Input"
PREPARED = os.path.dirname(hits[0])
print("prepared_dir =", PREPARED)
print("contents:", os.listdir(PREPARED))
for split in ("train", "validation", "test"):
    p = os.path.join(PREPARED, split)
    if os.path.isdir(p):
        print(f"  {split}: {len(os.listdir(p))} files")

In [ ]:
# 3. Build Kaggle config + generate card2vec / card_frequencies into writable /kaggle/working
#    (the attached prepared dataset is read-only, so aux files go to AUX and the
#     trainer reads them via card2vec_path / card_frequencies_path).
import sys, os, shutil, pathlib, yaml
REPO = "/kaggle/working/rigged-royale-matchup-ml"
sys.path.insert(0, f"{REPO}/src")
from rigged_matchup_ml.config import load_config
from rigged_matchup_ml.card2vec import pretrain_card_embeddings, write_card_frequencies

AUX = "/kaggle/working/prepared_aux"
os.makedirs(AUX, exist_ok=True)
cfg = yaml.safe_load(open(f"{REPO}/config/default.yaml"))
cfg["data"]["prepared_dir"] = PREPARED
cfg["training"]["device"] = "auto"             # picks cuda on Kaggle
cfg["training"]["artifact_dir"] = "/kaggle/working/artifacts"
cfg["training"]["num_workers"] = 2             # Kaggle box ~4 vCPU
cfg["training"]["card2vec_path"] = AUX         # read warm-start vectors from here
cfg["training"]["card_frequencies_path"] = AUX # read loss weights from here
# cfg["training"]["epochs"] = 10               # lower if you risk the 12h session cap
CONFIG_PATH = "/kaggle/working/kaggle.yaml"
yaml.safe_dump(cfg, open(CONFIG_PATH, "w"), sort_keys=False)
_cfg = load_config(pathlib.Path(CONFIG_PATH))

if cfg["training"].get("card2vec_init"):
    print("card2vec:", pretrain_card_embeddings(_cfg, output_dir=pathlib.Path(AUX)))
if cfg["training"].get("loss_card_frequency_weighting"):
    freq_in = os.path.join(PREPARED, "card_frequencies.json")
    if os.path.exists(freq_in):
        shutil.copy(freq_in, os.path.join(AUX, "card_frequencies.json"))
        print("card_frequencies: copied from prepared input")
    else:
        print("card_frequencies:", write_card_frequencies(_cfg, output_dir=pathlib.Path(AUX)))
print("wrote", CONFIG_PATH)

In [ ]:
# 3b. Sanity-check the new features are present in this checkout + aux files generated
import os
from rigged_matchup_ml import card_stats
from rigged_matchup_ml.dataset import encode_row

print("config max_elixir      :", cfg["model"].get("max_elixir"))
print("cross_heads            :", cfg["model"].get("cross_heads"))
print("use_deck_transformer   :", cfg["model"].get("use_deck_transformer"))
print("card2vec_init          :", cfg["training"].get("card2vec_init"))
print("freq weighting         :", cfg["training"].get("loss_card_frequency_weighting"))
assert cfg["model"].get("use_deck_transformer") is not None, "stale clone — pull latest"
assert cfg["model"].get("cross_heads"), "stale clone — pull latest"

print("card2vec.npy present   :", os.path.exists(os.path.join(AUX, "card2vec.npy")))
print("card_frequencies present:", os.path.exists(os.path.join(AUX, "card_frequencies.json")))

# encode_row still emits per-card elixir (derived from card ids).
demo = {
    "team_card_ids": [26000000, 26000004, 26000074, 26000010, 26000028, 26000003, 26000011, 26000064],
    "opponent_card_ids": [26000055, 26000011, 26000007, 26000012, 27000003, 28000001, 28000004, 26000018],
    "team_evolution_levels": [1, 0, 0, 0, 0, 0, 0, 0], "opponent_evolution_levels": [0] * 8,
    "team_hero_levels": [0] * 8, "opponent_hero_levels": [0] * 8,
    "team_card_roles": [1, 1, 2, 1, 1, 1, 1, 1], "opponent_card_roles": [1] * 8,
    "team_tower_troop_id": 0, "opponent_tower_troop_id": 0,
    "segment": "ladder:9000-11999", "patch": "2026-06", "matrix_prior": 0.5, "win": True,
}
enc = encode_row(demo, {"cards": {}, "towers": {}, "segments": {}, "patches": {}})
assert "team_elixir" in enc and "opponent_elixir" in enc, "elixir not wired — stale clone?"
print("OK: archetype + multi-head counters + card2vec + freq-weighting + elixir active.")

In [ ]:
# 4. Train (calls train_model directly — skips CLI, no db deps needed)
import sys, pathlib
sys.path.insert(0, f"{REPO}/src")
from rigged_matchup_ml.config import load_config
from rigged_matchup_ml.trainer import train_model
checkpoint = train_model(load_config(pathlib.Path(CONFIG_PATH)))
print("\nCHECKPOINT:", checkpoint)

In [ ]:
# 5. Show artifacts (download from the Output tab, or Save Version to persist)
import os
art = "/kaggle/working/artifacts"
for f in sorted(os.listdir(art)):
    print(f, round(os.path.getsize(os.path.join(art, f)) / 1e6, 2), "MB")

In [ ]:
# 6. (optional) Evaluate the trained checkpoint on the held-out test split
from rigged_matchup_ml.trainer import evaluate_checkpoint
import json
metrics = evaluate_checkpoint(load_config(pathlib.Path(CONFIG_PATH)), pathlib.Path(checkpoint))
print(json.dumps(metrics, indent=2))